# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import pandas as pd

file_path = r"../data/raw/historicalweeklydata.xlsx"

df_freddie_raw = pd.read_excel(
    file_path,
    header=None
)

print(df_freddie_raw.head(15))
print(df_freddie_raw.shape)

                      0      1       2  \
0                   NaN    NaN     NaN   
1                   NaN    NaN     NaN   
2                   NaN    NaN     NaN   
3                   NaN    NaN     NaN   
4                   NaN   U.S.   30 yr   
5                   NaN  30 yr  fees &   
6                  Week    FRM  points   
7   1971-04-02 00:00:00   7.33     NaN   
8   1971-04-09 00:00:00   7.31           
9   1971-04-16 00:00:00   7.31           
10  1971-04-23 00:00:00   7.31           
11  1971-04-30 00:00:00   7.29           
12  1971-05-07 00:00:00   7.38           
13  1971-05-14 00:00:00   7.42           
14  1971-05-21 00:00:00   7.44           

                                                   3       4     5        6  \
0                                                NaN     NaN   NaN      NaN   
1                    PRIMARY MORTGAGE MARKET SURVEY®     NaN   NaN      NaN   
2   Summary page with all rate types - U.S. averages     NaN   NaN      NaN   
3          

In [2]:
df_freddie = df_freddie_raw.iloc[7:, [0, 1]].copy()

df_freddie.columns = [
    "date",
    "mortgage_rate_30yr"
]

print(df_freddie.head())
print(df_freddie.tail())
print(df_freddie.shape)

                   date mortgage_rate_30yr
7   1971-04-02 00:00:00               7.33
8   1971-04-09 00:00:00               7.31
9   1971-04-16 00:00:00               7.31
10  1971-04-23 00:00:00               7.31
11  1971-04-30 00:00:00               7.29
                                                   date mortgage_rate_30yr
2899                                2026-09-03 00:00:00               6.71
2900                                2026-09-10 00:00:00               6.76
2901                                2026-09-17 00:00:00               6.95
2902                                2026-09-24 00:00:00               7.03
2903  Although Freddie Mac attempts to provide relia...                NaN
(2897, 2)


In [3]:
# Convert fields to appropriate data types
df_freddie["date"] = pd.to_datetime(
    df_freddie["date"],
    errors="coerce"
)

df_freddie["mortgage_rate_30yr"] = pd.to_numeric(
    df_freddie["mortgage_rate_30yr"],
    errors="coerce"
)

# Remove rows that are not actual observations
df_freddie = df_freddie.dropna(
    subset=["date", "mortgage_rate_30yr"]
).copy()

# Reset the dataframe index
df_freddie = df_freddie.reset_index(drop=True)

# Final validation
print(df_freddie.head())
print(df_freddie.tail())

print("\nData Types:")
print(df_freddie.dtypes)

print("\nDate Range:")
print("Earliest:", df_freddie["date"].min())
print("Latest:", df_freddie["date"].max())

print("\nMissing Values:")
print(df_freddie.isna().sum())

print("\nTotal Observations:")
print(len(df_freddie))

        date  mortgage_rate_30yr
0 1971-04-02                7.33
1 1971-04-09                7.31
2 1971-04-16                7.31
3 1971-04-23                7.31
4 1971-04-30                7.29
           date  mortgage_rate_30yr
2891 2026-08-27                6.66
2892 2026-09-03                6.71
2893 2026-09-10                6.76
2894 2026-09-17                6.95
2895 2026-09-24                7.03

Data Types:
date                  datetime64[ns]
mortgage_rate_30yr           float64
dtype: object

Date Range:
Earliest: 1971-04-02 00:00:00
Latest: 2026-09-24 00:00:00

Missing Values:
date                  0
mortgage_rate_30yr    0
dtype: int64

Total Observations:
2896


In [4]:
freddie_analysis = df_freddie[
    (df_freddie["date"].dt.year >= 2005) &
    (df_freddie["date"].dt.year <= 2025)
].copy()

print("\nFreddie Mac Analysis Period: 2005–2025")
print(freddie_analysis["mortgage_rate_30yr"].describe())

print("\nAdditional Mortgage Rate Statistics")
print("Median:", freddie_analysis["mortgage_rate_30yr"].median())
print("Variance:", freddie_analysis["mortgage_rate_30yr"].var())
print("Skewness:", freddie_analysis["mortgage_rate_30yr"].skew())

print("\nAnalysis Dataset Shape")
print(freddie_analysis.shape)


Freddie Mac Analysis Period: 2005–2025
count    1096.000000
mean        4.863832
std         1.271869
min         2.650000
25%         3.850000
50%         4.540000
75%         6.170000
max         7.790000
Name: mortgage_rate_30yr, dtype: float64

Additional Mortgage Rate Statistics
Median: 4.54
Variance: 1.6176506895977067
Skewness: 0.2614543200448702

Analysis Dataset Shape
(1096, 2)


In [5]:
print("\nDate Range")
print("Earliest:", freddie_analysis["date"].min())
print("Latest:", freddie_analysis["date"].max())

print("\nMissing Values")
print(freddie_analysis.isna().sum())


Date Range
Earliest: 2005-01-06 00:00:00
Latest: 2025-12-31 00:00:00

Missing Values
date                  0
mortgage_rate_30yr    0
dtype: int64
